# 📑 QLM Report
 
## 🔗 Links
- **[BRD QLM - TBD](tbd)**  
  Comprehensive documentation outlining the business requirements for this report.
  
- **[Story Ticket - tbd](tbd)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket](https://cepalhellas.atlassian.net/browse/ASK-3462)**  
  A summary of findings from UAT.
 
---


In [0]:
# %pip install openpyxl
# %restart_python

In [0]:
import os
import sys

dbutils.widgets.text("run_date", "")
dbutils.widgets.text("adhoc_run", '')

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/HoistInvestorReports/", 1)[0] 
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import load_configs, build_portfolio_filter

env = get_env_prefix()


In [0]:
from datetime import datetime
from pytz import timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

athens_tz = timezone('Europe/Athens')
adhoc_run = dbutils.widgets.get("adhoc_run")

if adhoc_run == '1':
    dbutils.widgets.text("run_date", '')
    run_date = dbutils.widgets.get("run_date")
else:
    run_date = spark.sql(f"SELECT MAX(BusinessDate) FROM `{env}-cdh`.primaryactive.exposure").collect()[0][0].strftime("%Y-%m-%d")
run_date_str = run_date.replace('-', '')
now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')


In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
import pandas as pd

In [0]:
## Portfolios In Scope
portfolioName = "('Cell', 'Orbit', 'Pearl Portfolio A', 'Mercury', 'Light')"    #All hoist portfolios

##### Views

In [0]:
migrationDate = spark.sql(f"""
          SELECT   min(cast(AssignmentDate as date))  AS migrationDate, 
          PortfolioName
          FROM `{env}-cdh`.primary.portfolio
          WHERE PortfolioName IN {portfolioName}
          GROUP BY PortfolioName
      """)
migrationDate.createOrReplaceTempView("migrationDate")

dexposure = spark.sql(f"""
         SELECT *,
         {run_date!r} AS RunDate
         FROM `{env}-cdh`.derivative.get_all_exposures_by_business_date(DATE({run_date!r}))           
         WHERE sourcesystem = 'profits'
         AND PortfolioName IN {portfolioName}
         """)
dexposure.createOrReplaceTempView("dexposure")    

primaryexposure = spark.sql(f"""
         select * from (
              SELECT * , rank() OVER (PARTITION BY ExposureKey ORDER BY BusinessDate desc ) rnk
              FROM `{env}-cdh`.primary.exposure         
              WHERE 1=1 
              and sourcesystem = 'profits'               
              AND PortfolioName IN {portfolioName}
              and BusinessDate <= {run_date!r}  
         ) where rnk = 1
         """)
primaryexposure.createOrReplaceTempView("primaryexposure")      

dparty = spark.sql(f"""
         select * from (
              select * , rank() OVER (PARTITION BY CDI ORDER BY BusinessDate desc ) rnk
              from `{env}-cdh`.derivative.dparty
              WHERE BusinessDate = {run_date!r}              
              AND PortfolioName IN {portfolioName}
         ) where rnk = 1
  """)
dparty.createOrReplaceTempView("dparty")

dlegalprocessstatusagreement = spark.sql(f"""
       select *
       from `{env}-cdh`.derivative.dlegalprocessstatusagreement             
       where BusinessDate = {run_date!r} 
       AND PortfolioName IN {portfolioName}
       """)
dlegalprocessstatusagreement.createOrReplaceTempView("dlegalprocessstatusagreement")

dlegalprocessstatusdebtor = spark.sql(f"""
       select *
       from `{env}-cdh`.derivative.dlegalprocessstatusdebtor                  
       where BusinessDate = {run_date!r} 
       AND PortfolioName IN {portfolioName}
       """)
dlegalprocessstatusdebtor.createOrReplaceTempView("dlegalprocessstatusdebtor")

assignment = spark.sql(f"""
        select *
        from `{env}-cdh`.primary.assignment a
        where 1=1 
        AND BusinessDate = {run_date!r} 
        AND PortfolioName IN {portfolioName}
        and sourcesystem = 'qca'
       """)
assignment.createOrReplaceTempView("assignment")

ExchangeRates = spark.sql(f"""
       select *
       from `{env}-cdh`.reference.exchangerates 
       where BusinessDate = {run_date!r} 
       """)
ExchangeRates.createOrReplaceTempView("ExchangeRates")

dsettlementexposure = spark.sql(f"""     
-- Frist prioritize active settlements, then fulfilled, and then inactive
        select * from (
            select * ,  
                     rank() over (partition by EXPOSUREKEY order by 
                            case when settlementstatus  IN ('IMPLEMENTED', 'PENDING FOR IMPLEMENTATION', 'Implemented','Partially Fulfilled','Pending Implementation', 'Implemented (Inactive SOR MGR)' ,'Pending Implementation (Inactive SOR MGR)' ,'Running' ,'Working (Inactive SOR MGR)' ,'approved' ,'running' ) then 1  -- active
                             when settlementstatus  IN ('FULFILLED', 'Fulfilled' ,'Partially Fulfilled' ,'Partially Fulfilled (Inactive SOR MGR)' ,'unknown' ) then 2  -- fulfilled
                             when settlementstatus  IN ('NOT FULFILLED', 'Cancelled' ,'Cancelled - Client Rejection' ,'Cancelled - Communication Failed' ,'Cancelled - Cosmos Migration' ,'Cancelled - GAIA Migration' ,'Cancelled - Galaxy Migration' ,'Cancelled - Internal Rejection' ,'Cancelled - Merged CDI' ,'Client Cancelled (Inactive SOR MGR)' ,'Internal Cancelled (Inactive SOR MGR)' ,'Non-Active (Inactive SOR MGR)' ,'Rejected (Inactive SOR MGR)' ,'For Approval (Inactive SOR MGR)' ,'Not Fulfilled' ,'Not Fulfilled (Inactive SOR MGR)' ,'Out of Collection'  ) then 3  -- inactive
                             else 4 end asc, 
                              isPrimarySettlementFlag desc,  implementationdate desc, CDI) as Rank
            from `p-cdh`.derivative.dsettlementexposure  -- qa table not available
            WHERE BusinessDate = {run_date!r}                                    
            AND PortfolioName IN {portfolioName}
        ) where rank = 1 
  """)
dsettlementexposure.createOrReplaceTempView("dsettlementexposure")

reassignments = spark.sql(f"""                              
        select distinct re.LoanID
        from `{env}-reportinputfiles`.investor.hoist_reassignments as re
          """)
reassignments.createOrReplaceTempView("reassignments")

##### LegalAction Perimeter

In [0]:

LegalAction = spark.sql(f"""
       select L.* from (
              select *, rank() OVER (PARTITION BY LegalActionCode ORDER BY LegalActionDate desc, LegalActionStepsTARTDate desc, LegalActionStepEndDate desc ) rnk
              from `{env}-cdh`.primary.legalaction   
              where BusinessDate = {run_date!r} 
              AND PortfolioName IN {portfolioName}
              and SourceSystem = 'arotron'
              AND ( (legalactiontype ='Αγωγή'and legaltype = 'Αγνώστου Διαμονής') OR
                    (legalactiontype ='Αγωγή'and legaltype = 'Ελλειψεις Συμβατικων') OR
                    (legalactiontype ='Αναγγελία'and legaltype = 'Πλειστηριασμό Ημών') OR
                    (legalactiontype ='Αναγγελία'and legaltype = 'Πλειστηριασμό Τρίτου') OR
                    (legalactiontype ='Β Επίδοση'and legaltype like 'Β%Επίδοση Προς Γνώση') OR
                    (legalactiontype ='Β Επίδοση'and legaltype = 'Διαταγή Πληρωμής') OR
                    (legalactiontype ='Διαταγή Πληρωμής'and legaltype = 'Καταγγελία Σύμβασης') OR
                    (legalactiontype ='Εγγραφή Βάρους'and legaltype = 'Αναγκαστική') OR
              --       (legalactiontype ='Εγγραφή Βάρους'and legaltype = 'Συναινετική') OR -- excluded after talking with pasisis
                    (legalactiontype ='Εξώδικο'and legaltype = 'Καταγγελία Σύμβασης') OR
                    (legalactiontype ='Επίδοση ΔΠ'and legaltype like 'A%Επίδοση Προς Γνώση') OR
                    (legalactiontype ='Επίδοση ΔΠ'and legaltype = 'Α Επίδοση με Επιταγή') OR
                    (legalactiontype ='Κατάσχεση κινητών-ακινήτων'and legaltype = 'Ακίνητα') OR
                    (legalactiontype ='Κατάσχεση μισθωμάτων-εις χείρας Τρίτου'and legaltype = 'Ες Χείρας Τρίτου') OR
                    (legalactiontype ='Πλειστηριασμός Ημών'and legaltype = 'Ακίνητης Ιδιοκτησίας - Ενυπόθηκο') OR
                    (legalactiontype ='Πλειστηριασμός Τρίτου'and legaltype = 'Ακίνητης Ιδιοκτησίας - Ενυπόθηκο') OR
                    (legalactiontype ='Πλειστηριασμός Τρίτου'and legaltype = 'Ακίνητης Ιδιοκτησίας - Μη ενυπόθηκο') )
       ) AS L 
       LEFT JOIN MigrationDate as md on L.PortfolioName = MD.PortfolioName
       where L.rnk = 1 AND (IFNULL(CAST(L.LegalActionDate AS DATE), '1900-01-01') >= CAST( md.MigrationDate AS DATE)
       OR IFNULL(CAST(L.CompletionResultDate AS DATE), '1900-01-01') >= CAST( md.MigrationDate AS DATE) OR 
       IFNULL(CAST(L.LegalActionStepStartDate AS DATE), '1900-01-01') >= CAST( md.MigrationDate AS DATE))
       """)
LegalAction.createOrReplaceTempView("LegalAction")

##### Aggregations

In [0]:
ContractInfo = spark.sql(f"""
        select distinct
            de.AgreementKey, 
            SUM(de.OnBalance / coalesce( ER.ExchangeRate, 1)) as OnBalanceEURAgreement,
            SUM(de.OutstandingTotalBalance  / coalesce( ER.ExchangeRate, 1)) as OutstandingTotalBalanceEURAgreement
        from dexposure as de
        LEFT JOIN ExchangeRates          AS ER ON dE.Currency = ER.OriginalCurrency
        where de.IsActiveFlag = true and de.BusinessDate = '{run_date}'      
        group by de.AgreementKey

        """)
ContractInfo.createOrReplaceTempView("ContractInfo")

ContractDebtors = spark.sql(f"""
        select distinct
            de.AgreementKey, 
            count(distinct de.CDI) as CountDebtors
        from primaryexposure as de 
        Where PartyRelationshipType <> 'Other'       
        group by de.AgreementKey

        """)
ContractDebtors.createOrReplaceTempView("ContractDebtors")

ActiveSettlement = spark.sql(f"""
        select  AgreementKey, SettlementStatus from(
            select de.AgreementKey,sett.exposurekey, sett.ImplementationDate , sett.SettlementStatus
                , rank() over (partition by de.AgreementKey order by 
                    case when settlementstatus  IN ('Implemented (Inactive SOR MGR)' ,'Pending Implementation (Inactive SOR MGR)' ,'Running' ,'Working (Inactive SOR MGR)' ,'approved' ,'running' ) then 1  -- active
                    when settlementstatus  IN ('Fulfilled' ,'Partially Fulfilled' ,'Partially Fulfilled (Inactive SOR MGR)' ,'unknown' )then 2  -- fulfilled
                    when settlementstatus  IN ('Cancelled' ,'Cancelled - Client Rejection' ,'Cancelled - Communication Failed' 'Cancelled - Cosmos Migration' ,'Cancelled - GAIA Migration' ,'Cancelled - Galaxy Migration' ,'Cancelled - InternalRejection' ,'Cancelled - Merged CDI' ,'Client Cancelled (Inactive SOR MGR)' ,'Internal Cancelled (Inactive SOR MGR)' 'Non-Active (Inactive SOR MGR)' ,'Rejected (Inactive SOR MGR)' ,'For Approval (Inactive SOR MGR)' ,'Not Fulfilled' 'Not Fulfilled (Inactive SOR MGR)' ,'Out of Collection'  ) then 3  -- inactive
                    else 4 end asc, 
                     isPrimarySettlementFlag desc,  implementationdate desc, de.exposurekey desc, sett.settlementKey) as rnk
            from dexposure as de
            inner join dsettlementexposure as sett on sett.exposurekey = right(de.exposurekey, len(de.exposurekey)-3) and de.portfolioname = sett.portfolioname
            where de.IsActiveFlag = true   
            and de.businessdate = '{run_date}'    )
        where rnk = 1

        """)
ActiveSettlement.createOrReplaceTempView("ActiveSettlement")

ChannelAssignmentAgreement = spark.sql(f"""
        select distinct
            de.AgreementKey, 
            ass.Channel,
            ass.ExternalCompanyFullName
        from dexposure as de 
        INNER JOIN assignment as ass on ass.ExposureKey = de.ExposureKeyQCAlpha AND ass.cdi = de.cdi
        and ass.AssignmentType = 'Channel Assignment'     
        where ass.Channel is not null or ass.ExternalCompanyFullName is not null
        """)
ChannelAssignmentAgreement.createOrReplaceTempView("ChannelAssignmentAgreement")

CollectionAssignmentAgreement = spark.sql(f"""
        select distinct
            de.AgreementKey, 
            ass.Channel,
            ass.ExternalCompanyFullName
        from dexposure as de 
        INNER JOIN assignment as ass on ass.exposurekey = de.ExposureKeyQCAlpha  AND ass.cdi = de.cdi
        and ass.AssignmentType = 'Collection Assignment'     
        where ass.Channel is not null or ass.ExternalCompanyFullName is not null
        """)
CollectionAssignmentAgreement.createOrReplaceTempView("CollectionAssignmentAgreement")


##### Final Selection

In [0]:
Legal = spark.sql(f"""
        select distinct    
            CAST('{run_date}' AS DATE) as BusinessDate,      
            act.PortfolioName,
            dexp.CDI CDIPrimary,    
            dpprim.LegalNameGR as PartyNamePrimary,
            dexp.PartyTaxRegistrationNumber as PartyTaxRegistrationNumberPrimary,            
            act.CDI CDIReceiver,        
            dprec.LegalNameGR as PartyNameReceiver,
            dprec.TaxRegistrationNumber as PartyTaxRegistrationNumberReceiver,   
            TAR.TargetLegalAction as TargetLegalActionPrimary,      -- requires update following ticket DBTC-1647
            act.AgreementKey,
            cast(act.BankAccount as decimal(19,2)) as LitigationAmount, 
            '' as PropertyKey,      -- requires update following ticket DBTC-1647
            act.LegalActionType, 
            act.LegalType as RefersTo, 
            act.LegalActionCode,
            act.LegalActionStepType as LegalPhase,
            cast(act.LegalActionStepEndDate as date) LegalPhaseEndDate,
            cast(act.LegalActionDate as date) LegalActionStartDate,
            CAST(act.LegalActionStepStartDate AS DATE) as LegalPhaseStartDate,
            CAST(act.CompletionResultDate AS DATE), 
            act.CompletionResult, 
            CASE WHEN cast(act.CompletionResultDate as date) is not null THEN 'Y' else 'N' END as CompletionIndicator, 
            CASE WHEN RA.settlementstatus  IN ('Implemented (Inactive SOR MGR)' ,'Pending Implementation (Inactive SOR MGR)' ,'Running' ,'Working (Inactive SOR MGR)' ,'approved' ,'running' ) then  TRUE else FALSE end as ActiveSettlement,
            ra.SettlementStatus,    
            act.LawyerPartyName LegalAssignment, 
            lpsd.OperationalLegalStatusLevel3 DebtorLPS,
            CAST(lpsd.LegalProcessStatusLastChangeDate AS DATE) AS DebtorLPSDate,
            lpsa.OperationalLegalStatusLevel3 AgreementLPS,
            CAST(lpsa.LegalProcessStatusLastChangeDate AS DATE) AS AgreementLPSDate,
            cast(CI.OnBalanceEURAgreement as decimal(18,2)) as OnBalanceEURAgreement,
            cast(CI.OutstandingTotalBalanceEURAgreement as decimal(18,2)) as OutstandingTotalBalanceEURAgreement,
            assch.Channel as AssignmentChannel,
            assch.ExternalCompanyFullName AS AssignmentExternalCompanyFullName,    
            asscol.Channel as CollectionChannel,
            asscol.ExternalCompanyFullName AS CollectionExternalCompanyFullName,  
            CAST(CD.CountDebtors AS INT) CountOfBeneficiaries 
        FROM legalaction as act
        INNER JOIN dexposure AS dexp on act.agreementkey = dexp.agreementkey 
        LEFT JOIN DPARTY AS DPPrim ON DPPrim.CDI = dexp.CDI
        LEFT JOIN DPARTY AS DPRec ON DPRec.CDI = act.CDI
        -- LEFT JOIN assignment as assch on assch.exposurekey = dexp.ExposureKeyQCAlpha and assch.cdi = dexp.cdi and assch.AssignmentType = 'Channel Assignment'
        -- LEFT JOIN assignment as asscol on asscol.exposurekey = dexp.ExposureKeyQCAlpha and asscol.cdi = dexp.cdi and asscol.AssignmentType = 'Collection Assignment'
        LEFT JOIN ChannelAssignmentAgreement as assch on assch.agreementkey = dexp.agreementkey 
        LEFT JOIN CollectionAssignmentAgreement as asscol on asscol.agreementkey = dexp.agreementkey 
        LEFT JOIN dlegalprocessstatusdebtor as lpsd on lpsd.cdi = dexp.cdi
        LEFT JOIN dlegalprocessstatusagreement as lpsa on lpsa.AgreementKey = dexp.AgreementKey
        LEFT JOIN ContractInfo AS CI on CI.AgreementKey = dexp.AgreementKey
        LEFT JOIN ContractDebtors AS CD ON CD.AgreementKey = dexp.AgreementKey
        LEFT JOIN ActiveSettlement AS ra on ra.AgreementKey = dexp.AgreementKey 
        LEFT JOIN `{env}-reportinputfiles`.investor.QLM_TargetLegalAction_extfile AS TAR ON TAR.PrimaryOwnerCDI = dexp.CDI      
        LEFT JOIN reassignments as re on re.LoanID = dexp.ExposureKey
        WHERE RE.LoanID IS NULL AND (dexp.JUSTIFICATIONCODE IS NULL OR dexp.JUSTIFICATIONCODE NOT IN (43055,43056))
        """)
Legal.createOrReplaceTempView("Legal")

In [0]:
target_table_path_qlm = f"`{env}-cdh-delivery`.investorreports.hoist_QLM"

spark.sql(f"""DELETE FROM {target_table_path_qlm} WHERE BusinessDate = '{run_date}' AND PortfolioName IN {portfolioName}""")

Legal = Legal.withColumn("IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")))

Legal.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(target_table_path_qlm) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_qlm} SYNC METADATA")